In [9]:
from pathlib import Path
import json
import shutil
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

code_file=Path("/kaggle/input/datasets/risha8750/phase2-core3/phase2_core.py")
assert code_file.exists()
shutil.copy2(code_file,"/kaggle/working/phase2_core.py")
sys.path.insert(0,"/kaggle/working")
from phase2_core import *

seed_everything(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type=="cuda"
torch.backends.cudnn.deterministic=False
torch.backends.cudnn.benchmark=True

{"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__}

{'gpu': 'Tesla T4', 'torch': '2.10.0+cu128'}

In [10]:
cache_root=Path("/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/phase2_cache")
assert cache_root.exists()
assert (cache_root/"index.csv").exists()
assert (cache_root/"cache_config.json").exists()

out_root=Path("/kaggle/working/phase2_outputs")
out_root.mkdir(parents=True,exist_ok=True)
corner_dir=out_root/"corner"
edge_dir=out_root/"edge"
mesh_dir=out_root/"meshes"
for path in [corner_dir,edge_dir,mesh_dir]:
    path.mkdir(parents=True,exist_ok=True)

train_set=RoofDataset(cache_root,"train",augment=True)
val_set=RoofDataset(cache_root,"val")
test_set=RoofDataset(cache_root,"test")

len(train_set),len(val_set),len(test_set)

(25579, 3170, 3235)

In [11]:
batch_size=6
workers=2
train_loader=DataLoader(train_set,batch_size=batch_size,shuffle=True,num_workers=workers,pin_memory=True,drop_last=True,persistent_workers=True)
val_loader=DataLoader(val_set,batch_size=batch_size,shuffle=False,num_workers=workers,pin_memory=True,persistent_workers=True)

batch=next(iter(train_loader))
assert batch["points"].shape[1:]==(2048,8)
assert torch.isfinite(batch["points"]).all()

{k:tuple(v.shape) for k,v in batch.items() if torch.is_tensor(v)}

{'points': (6, 2048, 8),
 'corners': (6, 64, 3),
 'corner_count': (6,),
 'edges': (6, 128, 2),
 'edge_count': (6,),
 'center': (6, 3),
 'scale': (6,)}

In [12]:
corner_model=CornerPointNet2().to(device)
corner_target_epochs=40
corner_radius_m=0.35
corner_lr=0.0001
corner_weight_decay=0.0001
corner_optimizer=torch.optim.AdamW(corner_model.parameters(),lr=corner_lr,weight_decay=corner_weight_decay)
corner_scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(corner_optimizer,T_max=corner_target_epochs,eta_min=0.00001)

edge_model=EdgeGraphNet().to(device)
edge_target_epochs=30
edge_optimizer=torch.optim.AdamW(edge_model.parameters(),lr=0.0003,weight_decay=0.0001)
edge_scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(edge_optimizer,T_max=edge_target_epochs,eta_min=0.00001)

sum(p.numel() for p in corner_model.parameters()),sum(p.numel() for p in edge_model.parameters())

(807940, 298497)

In [13]:
corner_path=Path("/kaggle/input/datasets/risha8750/phase2-outputs/phase2_outputs/corner/corner_last.pt")
edge_path=Path("/kaggle/input/datasets/risha8750/phase2-outputs/phase2_outputs/edge/edge_last.pt")

assert corner_path.exists()
assert edge_path.exists()

corner_state=torch.load(corner_path,map_location=device,weights_only=False)
corner_model.load_state_dict(corner_state["model"])
corner_optimizer.load_state_dict(corner_state["optimizer"])
corner_scheduler.load_state_dict(corner_state["scheduler"])
corner_history=corner_state["history"]
corner_start=corner_history[-1]["epoch"]+1

edge_state=torch.load(edge_path,map_location=device,weights_only=False)
edge_model.load_state_dict(edge_state["model"])
edge_optimizer.load_state_dict(edge_state["optimizer"])
edge_scheduler.load_state_dict(edge_state["scheduler"])
edge_history=edge_state["history"]
edge_start=edge_history[-1]["epoch"]+1

for parameter in corner_model.parameters():
    parameter.requires_grad=True
corner_model.train()

assert next(corner_model.parameters()).requires_grad==True
assert corner_start==5
assert edge_start==5
assert np.isfinite(corner_history[-1]["train_loss"])
assert np.isfinite(corner_history[-1]["val_loss"])
assert np.isfinite(edge_history[-1]["train_loss"])
assert np.isfinite(edge_history[-1]["val_loss"])

corner_epochs_this_run=15
edge_epochs_this_run=15

print(corner_start,corner_history[-1])
print(edge_start,edge_history[-1])
print(list(range(corner_start,min(corner_start+corner_epochs_this_run,corner_target_epochs))))
print(list(range(edge_start,min(edge_start+edge_epochs_this_run,edge_target_epochs))))

5 {'epoch': 4, 'train_loss': 0.07425638595048119, 'train_cls': 0.010277274243640848, 'train_offset': 0.0639791117237417, 'val_loss': 0.07238895552767564, 'val_cls': 0.009904352453910402, 'val_offset': 0.06248460300149226, 'lr': 9.65745789630079e-05}
5 {'epoch': 4, 'train_loss': 0.022577089212662323, 'val_loss': 0.022269573072059675, 'lr': 9.397114317029975e-05}
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [14]:
def corner_epoch(loader,training):
    corner_model.train(training)
    totals=np.zeros(4,dtype=np.float64)
    context=torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            points=batch["points"].to(device,non_blocking=True).float()
            corners=batch["corners"].to(device,non_blocking=True).float()
            counts=batch["corner_count"].to(device,non_blocking=True)
            scales=batch["scale"].to(device,non_blocking=True).float()
            labels,offset_targets=corner_targets(points,corners,counts,scales,corner_radius_m)
            if training:
                corner_optimizer.zero_grad(set_to_none=True)
            logits,pred_offsets,_=corner_model(points)
            loss,cls_loss,reg_loss=corner_loss(logits,offset_targets,pred_offsets,labels)
            if not torch.isfinite(loss):
                raise RuntimeError("non-finite corner loss")
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(corner_model.parameters(),1.0)
                corner_optimizer.step()
            size=len(points)
            totals+=np.array([loss.item()*size,cls_loss.item()*size,reg_loss.item()*size,size])
    return {"loss":totals[0]/totals[3],"cls":totals[1]/totals[3],"offset":totals[2]/totals[3]}

In [15]:
class _NoOpScaler:
    def state_dict(self):
        return {}
    def load_state_dict(self,state):
        pass

corner_scaler=_NoOpScaler()

corner_stop=min(corner_start+corner_epochs_this_run,corner_target_epochs)
assert corner_stop==20
for epoch in range(corner_start,corner_stop):
    train_metrics=corner_epoch(train_loader,True)
    val_metrics=corner_epoch(val_loader,False)
    assert np.isfinite(train_metrics["loss"])
    assert np.isfinite(val_metrics["loss"])
    corner_scheduler.step()
    row={"epoch":epoch,**{f"train_{k}":v for k,v in train_metrics.items()},**{f"val_{k}":v for k,v in val_metrics.items()},"lr":corner_optimizer.param_groups[0]["lr"]}
    corner_history.append(row)
    state={"epoch":epoch,"model":corner_model.state_dict(),"optimizer":corner_optimizer.state_dict(),"scheduler":corner_scheduler.state_dict(),"scaler":corner_scaler.state_dict(),"history":corner_history}
    torch.save(state,corner_dir/"corner_last.pt")
    if val_metrics["loss"]<=min(x["val_loss"] for x in corner_history):
        torch.save(state,corner_dir/"corner_best.pt")

assert (corner_dir/"corner_last.pt").exists()
assert corner_history[-1]["epoch"]==19
pd.DataFrame(corner_history).tail(20)

,epoch,train_loss,train_cls,train_offset,val_loss,val_cls,val_offset,lr
0,0,0.096322,0.015950,0.080372,0.084671,0.012523,0.072148,0.000100
1,1,0.082868,0.012020,0.070848,0.080211,0.011496,0.068715,0.000099
2,2,0.079350,0.011307,0.068043,0.077524,0.011054,0.066471,0.000099
3,3,0.076686,0.010830,0.065856,0.074963,0.010536,0.064428,0.000098
4,4,0.074256,0.010277,0.063979,0.072389,0.009904,0.062485,0.000097
5,5,0.070960,0.009722,0.061238,0.067709,0.009070,0.058639,0.000095
6,6,0.066844,0.008956,0.057889,0.064736,0.008545,0.056191,0.000093
7,7,0.064471,0.008513,0.055958,0.063028,0.008142,0.054886,0.000091
8,8,0.062833,0.008230,0.054603,0.061116,0.007818,0.053297,0.000089
9,9,0.061542,0.008029,0.053513,0.060216,0.007825,0.052391,0.000087


In [16]:
def edge_epoch(loader,training):
    edge_model.train(training)
    total_loss=0.0
    total_graphs=0
    context=torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            points=batch["points"].to(device,non_blocking=True).float()
            corners=batch["corners"].to(device,non_blocking=True).float()
            counts=batch["corner_count"].to(device,non_blocking=True)
            edges=batch["edges"].to(device,non_blocking=True)
            edge_counts=batch["edge_count"].to(device,non_blocking=True)
            with torch.no_grad():
                _,_,features=corner_model(points)
            if training:
                edge_optimizer.zero_grad(set_to_none=True)
            losses=[]
            for i in range(len(points)):
                n=int(counts[i])
                xyz=corners[i,:n]
                score=torch.ones(n,device=device)
                node_features=interpolate_features(points[i],features[i],xyz.detach().cpu().numpy())
                pairs,logits=edge_model(xyz,score,node_features)
                labels=edge_labels(pairs,edges[i],int(edge_counts[i]))
                losses.append(focal_binary(logits,labels,0.75,2.0))
            loss=torch.stack(losses).mean()
            if not torch.isfinite(loss):
                raise RuntimeError("non-finite edge loss")
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(edge_model.parameters(),1.0)
                edge_optimizer.step()
            total_loss+=loss.item()*len(points)
            total_graphs+=len(points)
    return total_loss/max(total_graphs,1)

In [ ]:
for parameter in corner_model.parameters():
    parameter.requires_grad=False
corner_model.eval()

assert next(corner_model.parameters()).requires_grad==False

edge_scaler=_NoOpScaler()

edge_stop=min(edge_start+edge_epochs_this_run,edge_target_epochs)
assert edge_stop==20
for epoch in range(edge_start,edge_stop):
    train_loss=edge_epoch(train_loader,True)
    val_loss=edge_epoch(val_loader,False)
    assert np.isfinite(train_loss)
    assert np.isfinite(val_loss)
    edge_scheduler.step()
    row={"epoch":epoch,"train_loss":train_loss,"val_loss":val_loss,"lr":edge_optimizer.param_groups[0]["lr"]}
    edge_history.append(row)
    state={"epoch":epoch,"model":edge_model.state_dict(),"optimizer":edge_optimizer.state_dict(),"scheduler":edge_scheduler.state_dict(),"scaler":edge_scaler.state_dict(),"history":edge_history}
    torch.save(state,edge_dir/"edge_last.pt")
    if val_loss<=min(x["val_loss"] for x in edge_history):
        torch.save(state,edge_dir/"edge_best.pt")

assert (edge_dir/"edge_last.pt").exists()
assert edge_history[-1]["epoch"]==19
pd.DataFrame(edge_history).tail(20)